# Error Analysis

Notebook ini digunakan untuk menganalisis kesalahan prediksi model CNN.

Tujuan:

- Mengidentifikasi sampel yang salah diprediksi.
- Mengetahui pola kesalahan model.
- Menentukan rekomendasi perbaikan model.

In [1]:
# ==========================================================
# IMPORT LIBRARY
# ==========================================================

import os

import numpy as np

import pandas as pd

from tensorflow.keras.models import load_model

In [2]:
# ==========================================================
# LOAD VALIDATION DATASET
# ==========================================================

X = np.load(

    "X_validation.npy"

)

y = np.load(

    "y_validation.npy"

)

print("Validation dataset loaded successfully.")

Validation dataset loaded successfully.


In [3]:
# ==========================================================
# LABEL NAME
# ==========================================================

label_names = [

    "Anger",

    "Sadness",

    "Neutral",

    "Happiness"

]

In [4]:
# ==========================================================
# LOAD MODEL
# ==========================================================

model = load_model(

    "ser_model_augmented.keras"

)

print("Model loaded successfully.")

Model loaded successfully.


In [5]:
# ==========================================================
# MODEL PREDICTION
# ==========================================================

prediction = model.predict(

    X,

    verbose=1

)

y_pred = np.argmax(

    prediction,

    axis=1

)

44/44 ━━━━━━━━━━━━━━━━━━━━ 5s 107ms/step


In [6]:
# ==========================================================
# BUILD RESULT TABLE
# ==========================================================

result_df = pd.DataFrame({

    "Actual": y,

    "Predicted": y_pred

})

result_df["Actual Emotion"] = result_df["Actual"].map(

    dict(

        enumerate(label_names)

    )

)

result_df["Predicted Emotion"] = result_df["Predicted"].map(

    dict(

        enumerate(label_names)

    )

)

result_df["Correct"] = (

    result_df["Actual"]

    ==

    result_df["Predicted"]

)

result_df.head()

,Actual,Predicted,Actual Emotion,Predicted Emotion,Correct
0,1,2,Sadness,Neutral,False
1,2,2,Neutral,Neutral,True
2,2,1,Neutral,Sadness,False
3,1,2,Sadness,Neutral,False
4,3,3,Happiness,Happiness,True


In [7]:
# ==========================================================
# WRONG PREDICTION
# ==========================================================

wrong_prediction = result_df[

    result_df["Correct"] == False

]

wrong_prediction.head(20)

,Actual,Predicted,Actual Emotion,Predicted Emotion,Correct
0,1,2,Sadness,Neutral,False
2,2,1,Neutral,Sadness,False
3,1,2,Sadness,Neutral,False
6,3,0,Happiness,Anger,False
7,0,1,Anger,Sadness,False
8,0,3,Anger,Happiness,False
11,2,1,Neutral,Sadness,False
13,0,3,Anger,Happiness,False
14,0,3,Anger,Happiness,False
19,2,1,Neutral,Sadness,False


In [8]:
# ==========================================================
# TOTAL WRONG PREDICTION
# ==========================================================

print(

    "Total Wrong Prediction :",

    len(

        wrong_prediction

    )

)

print(

    "Total Correct Prediction :",

    len(

        result_df

    )

    -

    len(

        wrong_prediction

    )

)

Total Wrong Prediction : 509
Total Correct Prediction : 868


In [9]:
# ==========================================================
# PREDICTION CONFIDENCE
# ==========================================================

result_df["Confidence"] = np.max(

    prediction,

    axis=1

) * 100

result_df["Confidence"] = result_df["Confidence"].round(2)

result_df.head()

,Actual,Predicted,Actual Emotion,Predicted Emotion,Correct,Confidence
0,1,2,Sadness,Neutral,False,58.590000
1,2,2,Neutral,Neutral,True,92.650002
2,2,1,Neutral,Sadness,False,61.049999
3,1,2,Sadness,Neutral,False,72.900002
4,3,3,Happiness,Happiness,True,47.730000


In [10]:
# ==========================================================
# WRONG PREDICTION WITH CONFIDENCE
# ==========================================================

wrong_prediction = result_df[

    result_df["Correct"] == False

]

wrong_prediction = wrong_prediction.sort_values(

    by="Confidence",

    ascending=False

)

wrong_prediction.head(20)

,Actual,Predicted,Actual Emotion,Predicted Emotion,Correct,Confidence
876,3,2,Happiness,Neutral,False,97.949997
1368,3,2,Happiness,Neutral,False,97.669998
126,3,2,Happiness,Neutral,False,97.660004
949,3,2,Happiness,Neutral,False,97.199997
1102,3,1,Happiness,Sadness,False,95.690002
1312,2,1,Neutral,Sadness,False,92.800003
1091,3,2,Happiness,Neutral,False,92.639999
340,3,1,Happiness,Sadness,False,92.540001
593,1,2,Sadness,Neutral,False,91.599998
713,3,2,Happiness,Neutral,False,91.139999


In [11]:
# ==========================================================
# BUILD DATASET METADATA
# ==========================================================

dataset_path = "Dataset_Final_Indonesia"

emotion_folders = {

    "01_Anger": "Anger",

    "02_Sadness": "Sadness",

    "03_Neutral": "Neutral",

    "04_Happiness": "Happiness"

}

dataset_metadata = []

for folder, emotion in emotion_folders.items():

    folder_path = os.path.join(

        dataset_path,

        folder

    )

    audio_files = sorted(

        os.listdir(folder_path)

    )

    for file_name in audio_files:

        dataset_metadata.append({

            "Filename": file_name,

            "Emotion": emotion,

            "Folder": folder,

            "Path": os.path.join(

                folder_path,

                file_name

            )

        })

dataset_metadata = pd.DataFrame(

    dataset_metadata
)

dataset_metadata

,Filename,Emotion,Folder,Path
0,anger001.wav,Anger,01_Anger,Dataset_Final_Indonesia\01_Anger\anger001.wav
1,anger002.wav,Anger,01_Anger,Dataset_Final_Indonesia\01_Anger\anger002.wav
2,anger003.wav,Anger,01_Anger,Dataset_Final_Indonesia\01_Anger\anger003.wav
3,anger004.wav,Anger,01_Anger,Dataset_Final_Indonesia\01_Anger\anger004.wav
4,anger005.wav,Anger,01_Anger,Dataset_Final_Indonesia\01_Anger\anger005.wav
...,...,...,...,...
1716,happiness456.wav,Happiness,04_Happiness,Dataset_Final_Indonesia\04_Happiness\happiness...
1717,happiness457.wav,Happiness,04_Happiness,Dataset_Final_Indonesia\04_Happiness\happiness...
1718,happiness458.wav,Happiness,04_Happiness,Dataset_Final_Indonesia\04_Happiness\happiness...
1719,happiness459.wav,Happiness,04_Happiness,Dataset_Final_Indonesia\04_Happiness\happiness...


In [12]:
# ==========================================================
# DATASET INFORMATION
# ==========================================================

print(

    "Total Audio :", 

    len(dataset_metadata)

)

print()

print(

    dataset_metadata.groupby(

        "Emotion"

    ).size()

)

Total Audio : 1721

Emotion
Anger        400
Happiness    460
Neutral      461
Sadness      400
dtype: int64
